# 06_attn_mask

『밑바닥부터 시작하는 딥러닝 ❻』 실습 코드 — 원본: `ch02/06_attn_mask.py`

셀을 위에서부터 차례대로 실행하세요.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class Attention(nn.Module):
    def __init__(self, embed_dim, key_dim):
        super().__init__()
        # Q, K, V를 생성하는 변환 행렬
        self.W_q = nn.Linear(embed_dim, key_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, key_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)

        self.key_dim = key_dim

    def forward(self, x):  # x: (B, C, E)
        Q = self.W_q(x)    # Q: (B, C, D)
        K = self.W_k(x)    # K: (B, C, D)
        V = self.W_v(x)    # V: (B, C, E)

        # 어텐션 점수 계산
        K_t = K.transpose(-2, -1)      # (B, D, C)
        scores = torch.matmul(Q, K_t)  # (B, C, C)
        scores = scores / (self.key_dim ** 0.5)

        # 마스크 적용
        B, C, E = x.shape
        mask = torch.tril(torch.ones(C, C, device=scores.device))
        scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)

        output = torch.matmul(weights, V)  # (B, C, E)
        return output

In [ ]:
attention = Attention(embed_dim=256, key_dim=64)
x = torch.randn(2, 5, 256)  # (batch_size=2, context_len=5, embed_dim=256)
y = attention(x)

In [ ]:
print("入力形状:", x.shape)
print("出力形状:", y.shape)